# 05-02 工具调用与 ReAct Agent

**Tool Use 是 LLM 变成 Agent 的关键能力**：模型不再只输出文本，还能调用外部函数。

**本节目标**：
- 用 @tool 装饰器定义工具
- 理解 Tool Schema（JSON Schema → LLM 知道如何调用）
- 手动实现 ReAct 循环
- 使用 AgentExecutor

---

In [ ]:
import os, sys, json
sys.path.insert(0, "..")
from dotenv import load_dotenv
load_dotenv("../.env")

try:
    from langchain_core.tools import tool
    from langchain_openai import ChatOpenAI
    HAS_LC = True
    print("LangChain 导入成功")
except ImportError:
    HAS_LC = False
    print("LangChain 未安装，将展示概念代码")

## 1. 定义工具

In [ ]:
# 模拟广告平台数据
AD_PERFORMANCE_DB = {
    "ad_1": {"title": "游戏皮肤限时折扣", "impressions": 50000, "clicks": 1500, "cost": 750.0, "converts": 120},
    "ad_2": {"title": "美妆新品体验", "impressions": 30000, "clicks": 600, "cost": 480.0, "converts": 45},
    "ad_3": {"title": "在线编程课程", "impressions": 20000, "clicks": 400, "cost": 200.0, "converts": 80},
}

if HAS_LC:
    @tool
    def get_ad_performance(ad_id: str) -> str:
        """获取广告的投放效果数据，包括展示量、点击量、消耗、转化数"""
        data = AD_PERFORMANCE_DB.get(ad_id)
        if not data:
            return f"未找到广告 {ad_id}"
        ctr = 100 * data["clicks"] / data["impressions"]
        cvr = 100 * data["converts"] / data["clicks"]
        return json.dumps({
            **data, "ctr_pct": round(ctr, 2), "cvr_pct": round(cvr, 2)
        }, ensure_ascii=False)

    @tool
    def check_ad_compliance(content: str) -> str:
        """检查广告内容是否合规（极限词、虚假宣传等）"""
        issues = []
        forbidden = ["最", "第一", "绝对", "100%"]
        for word in forbidden:
            if word in content:
                issues.append(f"含极限词'{word}'")
        return json.dumps({"compliant": len(issues) == 0, "issues": issues}, ensure_ascii=False)

    @tool
    def calculate_roi(ad_id: str, revenue_per_convert: float = 100.0) -> str:
        """计算广告ROI（投资回报率）"""
        data = AD_PERFORMANCE_DB.get(ad_id)
        if not data:
            return f"未找到广告 {ad_id}"
        revenue = data["converts"] * revenue_per_convert
        roi = (revenue - data["cost"]) / data["cost"] * 100
        return f"收入={revenue}元, 成本={data['cost']}元, ROI={roi:.1f}%"

    tools = [get_ad_performance, check_ad_compliance, calculate_roi]
    
    # 查看 Tool Schema（这就是发给 LLM 的函数定义）
    print("Tool Schema 示例:")
    print(json.dumps(get_ad_performance.args_schema.schema(), indent=2, ensure_ascii=False))
else:
    print("""
@tool 装饰器会自动：
  1. 从函数签名提取参数类型 → JSON Schema
  2. 从 docstring 提取描述 → 告诉 LLM 这个工具干什么
  3. 这个 Schema 发给 LLM 后，LLM 就知道可以调用什么工具、传什么参数
    """)

## 2. ReAct 循环（手动实现）

ReAct = **Re**asoning + **Act**ing，LLM 交替进行思考和行动：
```
Thought: 用户想知道广告效果，我需要先查数据
Action: get_ad_performance("ad_1")
Observation: {clicks: 1500, ctr: 3.0%...}
Thought: CTR 3% 高于均值，让我再算下 ROI
Action: calculate_roi("ad_1")
Observation: ROI=1500%
Thought: 数据齐了，可以给出建议
Final Answer: 广告效果很好...
```

In [ ]:
# 手动实现 ReAct 循环
def manual_react_agent(query: str, tools_dict: dict, max_steps: int = 5):
    """手动 ReAct Agent（理解原理用）"""
    from utils.llm_client import call_llm
    
    tool_descriptions = "\n".join(
        f"- {name}: {func.__doc__}" for name, func in tools_dict.items()
    )
    
    messages = []
    system = f"""你是B站广告分析助手。可用工具:\n{tool_descriptions}\n
回答时用以下格式:
Thought: (你的思考)
Action: tool_name("arg")
或者当你准备好最终答案时:
Final Answer: (最终回答)"""
    
    current_input = query
    context = ""
    
    for step in range(max_steps):
        try:
            response = call_llm(
                f"{context}\n\n用户问题: {current_input}" if context else current_input,
                system=system,
                max_tokens=300
            )
        except Exception:
            response = f"Thought: 我需要查看广告数据\nAction: get_ad_performance(\"ad_1\")" if step == 0 else "Final Answer: 基于查询到的数据分析..."
        
        print(f"\n--- Step {step+1} ---")
        print(response)
        
        if "Final Answer:" in response:
            return response.split("Final Answer:")[-1].strip()
        
        # 解析 Action 调用
        if "Action:" in response:
            action_line = response.split("Action:")[-1].strip().split("\n")[0]
            # 简单解析工具调用
            for name, func in tools_dict.items():
                if name in action_line:
                    import re
                    args = re.findall(r'"([^"]+)"', action_line)
                    try:
                        result = func(args[0]) if args else func("ad_1")
                        observation = f"Observation: {result}"
                    except Exception as e:
                        observation = f"Observation: 工具调用失败: {e}"
                    print(observation)
                    context += f"\n{response}\n{observation}"
                    break
    
    return "达到最大步数，无法完成分析"

# 定义简化版工具函数（不用 @tool 装饰器）
def get_perf(ad_id: str) -> str:
    """获取广告效果数据"""
    data = AD_PERFORMANCE_DB.get(ad_id, {})
    if data:
        ctr = 100 * data["clicks"] / data["impressions"]
        return json.dumps({**data, "ctr_pct": round(ctr, 2)}, ensure_ascii=False)
    return "未找到"

print("=== 手动 ReAct Agent ===")
result = manual_react_agent(
    "分析ad_1的投放效果，给出优化建议",
    {"get_ad_performance": get_perf}
)
print(f"\n最终答案: {result}")

## 3. LangChain AgentExecutor

In [ ]:
if HAS_LC and os.environ.get("OPENAI_API_KEY"):
    from langchain.agents import create_tool_calling_agent, AgentExecutor
    from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
    
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "你是B站广告分析助手，帮助广告主分析投放效果并给出优化建议。"),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ])
    
    agent = create_tool_calling_agent(llm, tools, prompt)
    executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=5)
    
    result = executor.invoke({"input": "查看ad_1和ad_3的效果，告诉我哪个ROI更高"})
    print(f"\n结果: {result['output']}")
else:
    print("""
AgentExecutor 工作流程：
  1. 用户输入 → prompt 组装 → LLM 推理
  2. LLM 输出 tool_calls → 执行工具 → 获得 Observation
  3. Observation 加入 agent_scratchpad → 继续推理
  4. 重复 2-3 直到 LLM 输出 Final Answer 或达到 max_iterations

关键参数：
  verbose=True         → 打印每步的 Thought/Action/Observation
  max_iterations=5     → 防止死循环
  handle_parsing_errors=True → 工具调用格式错误时自动重试
    """)

## ReAct vs Function Calling vs Plan-and-Execute

| 模式 | 原理 | 优点 | 缺点 |
|------|------|------|------|
| ReAct | 逐步 Thought→Action→Obs | 灵活，可纠错 | 步骤多，token 消耗大 |
| Function Calling | LLM 直接输出函数调用 JSON | 格式准确，速度快 | 依赖模型原生支持 |
| Plan-and-Execute | 先规划全部步骤，再逐步执行 | 复杂任务效果好 | 规划错误传播 |

## 面试速记

| 问题 | 要点 |
|------|------|
| Tool Schema 的作用 | 告诉 LLM 有哪些工具、参数类型和描述，LLM 才能正确调用 |
| 并行 Tool Calling | GPT-4o 支持一次输出多个 tool_calls，并行执行节省时间 |
| Agent 什么时候停止 | LLM 不再输出 tool_calls（认为已完成） 或达到 max_iterations |

**下一节**: `03_langgraph_state_graph.ipynb`